# Interview data visualizations

5 visualizations built in Vega-Altair, with Tableau recreation instructions for each.

**Prerequisites:** Run from the same project folder containing your `output/` directory.

In [1]:
!pip install altair vl-convert-python pandas openpyxl -q

In [2]:
import altair as alt
import pandas as pd
import numpy as np
import os

alt.data_transformers.disable_max_rows()

# Load all data
ca = pd.read_excel("output/coded_answers_long.xlsx")
tp = pd.read_excel("output/theme_prevalence.xlsx")
pp = pd.read_excel("output/participant_profiles.xlsx")
sh = pd.read_excel("output/sentiment_heatmap.xlsx", index_col=0)
co = pd.read_excel("output/theme_cooccurrence_long.xlsx")
ptm = pd.read_excel("output/participant_theme_matrix_long.xlsx")

# Clean participant names
def short_name(s):
    return s.replace(".txt", "").replace("Interview ", "")

print("Data loaded.")

Data loaded.


---
## Viz 1: Paired bar chart — themes felt vs. barriers named

Shows the gap between what participants *feel* across all questions vs. what they *named* as their one barrier to remove (Q16).

### Tableau instructions
1. Create a new data source from the CSV this cell exports (`viz_paired_barriers.csv`)
2. Drag `theme` to **Rows**
3. Drag `count` to **Columns**
4. Drag `type` to **Color**
5. Right-click `theme` on Rows -> Sort -> By Field -> `count` -> Descending (select the 'Themes felt' type)
6. Set mark type to **Bar** (should be default)
7. Under Color, choose two distinct colors: e.g. blue for 'Themes felt', coral/orange for 'Barrier named'
8. Right-click the x-axis -> Edit Axis -> set range from 0 to 8
9. Title: "What participants feel vs. what actually stops them"

In [3]:
# Build paired data
felt = tp[["theme", "participants_mentioning"]].copy()
felt.columns = ["theme", "count"]
felt["type"] = "Themes felt (all questions)"

# Extract barrier themes from Q16
barrier_q = ca[ca["question"].str.contains("barrier", case=False)]
barrier_counts = {}
for _, row in barrier_q.iterrows():
    for t in str(row["themes"]).split(", "):
        clean = t.strip().replace("_", " ").title()
        if clean and clean != "Nan":
            barrier_counts[clean] = barrier_counts.get(clean, 0) + 1

named = pd.DataFrame([{"theme": k, "count": v} for k, v in barrier_counts.items()])
named["type"] = "Barrier named (Q16)"

# Fill missing themes with 0 in each group
all_themes = set(felt["theme"].tolist()) | set(named["theme"].tolist())
for t in all_themes:
    if t not in felt["theme"].values:
        felt = pd.concat([felt, pd.DataFrame([{"theme": t, "count": 0, "type": felt["type"].iloc[0]}])])
    if t not in named["theme"].values:
        named = pd.concat([named, pd.DataFrame([{"theme": t, "count": 0, "type": named["type"].iloc[0]}])])

paired = pd.concat([felt, named], ignore_index=True)

# Sort order based on 'felt' count
theme_order = felt.sort_values("count", ascending=False)["theme"].tolist()

chart1 = alt.Chart(paired).mark_bar().encode(
    y=alt.Y("theme:N", sort=theme_order, title=None),
    x=alt.X("count:Q", title="Number of participants", scale=alt.Scale(domain=[0, 8])),
    color=alt.Color("type:N", title="",
        scale=alt.Scale(range=["#378ADD", "#D85A30"]),
    ),
    yOffset=alt.YOffset("type:N"),
    tooltip=["theme", "type", "count"]
).properties(
    width=500,
    height=450,
    title="What participants feel vs. what actually stops them"
)

# Export for Tableau
os.makedirs("viz_exports", exist_ok=True)
paired.to_csv("viz_exports/viz1_paired_barriers.csv", index=False)

chart1

alt.Chart(...)

---
## Viz 2: Sentiment heatmap

Questions x participants, colored by sentiment score. Shows universal agreement (all green/red rows) and divergence.

### Tableau instructions
1. Open `viz2_sentiment_long.csv`
2. Drag `question_short` to **Rows**
3. Drag `participant_short` to **Columns**
4. Drag `sentiment` to **Color**
5. Change mark type to **Square**
6. Click Color -> Edit Colors -> choose **Red-Green Diverging** palette
7. Set center to 0, range from -2 to 2
8. Drag `sentiment` to **Label** as well (so the number shows in each cell)
9. Right-click `question_short` on Rows -> Sort -> Manual, then reorder to match interview order (Q1-Q18)

In [5]:
# Reshape sentiment heatmap to long format
sent_long = sh.reset_index().melt(id_vars="question", var_name="participant", value_name="sentiment")
sent_long = sent_long.dropna(subset=["sentiment"])
sent_long["participant_short"] = sent_long["participant"].apply(short_name)

# Create short question labels with Q number
q_map = {}
for q in sent_long["question"].unique():
    num = q.split(".")[0] if "." in q else q[:4]
    short = q.split(".", 1)[1].strip()[:45] + "..." if len(q) > 50 else q.split(".", 1)[1].strip()
    q_map[q] = f"{num}. {short}"

sent_long["question_short"] = sent_long["question"].map(q_map)

# Get question order
q_order = [q_map[q] for q in sh.index if q in q_map]
# Remove duplicates while preserving order
seen = set()
q_order_unique = []
for q in q_order:
    if q not in seen:
        q_order_unique.append(q)
        seen.add(q)

chart2 = alt.Chart(sent_long).mark_rect(stroke="white", strokeWidth=1).encode(
    y=alt.Y("question_short:N", sort=q_order_unique, title=None),
    x=alt.X("participant_short:N", title=None),
    color=alt.Color("sentiment:Q",
        scale=alt.Scale(domain=[-2, -1, 0, 1, 2],
                        range=["#E24B4A", "#F09595", "#F1EFE8", "#97C459", "#639922"]),
        title="Sentiment"
    ),
    tooltip=["participant_short", "question_short", "sentiment"]
).properties(
    width=400,
    height=500,
    title="Sentiment across all participants and questions"
)

text2 = alt.Chart(sent_long).mark_text(fontSize=10).encode(
    y=alt.Y("question_short:N", sort=q_order_unique),
    x=alt.X("participant_short:N"),
    text=alt.Text("sentiment:Q", format=".0f"),
    color=alt.Color("sentiment:Q",
        scale=alt.Scale(domain=[-2, 0, 2], range=["#501313", "#444441", "#173404"]),
        legend=None
    )
)

viz2 = (chart2 + text2)

sent_long.to_csv("viz_exports/viz2_sentiment_long.csv", index=False)
viz2

alt.LayerChart(...)

---
## Viz 3: Dot strip plot — sentiment spread on key questions

Shows individual participant positions on Q3 (social media feelings) and Q7 (body relationship) — the two most divergent questions.

### Tableau instructions
1. Open `viz3_sentiment_strip.csv`
2. Drag `question_short` to **Rows**
3. Drag `sentiment` to **Columns** (make sure it's set to Dimension, not measure — right-click -> Dimension)
4. Actually, keep as measure with AGG set to MIN or use it on Detail
5. Better approach: Drag `sentiment` to Columns as continuous. Drag `participant_short` to **Detail**
6. Change mark type to **Circle**, increase size
7. Drag `participant_short` to **Color**
8. Drag `participant_short` to **Label**
9. Edit x-axis: fixed range -2 to 2, add a reference line at 0
10. Add a reference line: right-click axis -> Add Reference Line -> Constant value = 0, gray dashed

In [6]:
# Get Q3 and Q7 data
q3 = ca[ca["question"].str.startswith("Q3.")].copy()
q7 = ca[ca["question"].str.startswith("Q7.")].copy()

q3["question_short"] = "Q3: Social media feelings"
q7["question_short"] = "Q7: Relationship with body"

strip_data = pd.concat([q3, q7], ignore_index=True)
strip_data["participant_short"] = strip_data["participant"].apply(short_name)
strip_data = strip_data.dropna(subset=["sentiment"])

# Jitter y slightly to avoid overlap
np.random.seed(42)
strip_data["jitter"] = np.random.uniform(-0.15, 0.15, len(strip_data))

points = alt.Chart(strip_data).mark_circle(size=120, opacity=0.85).encode(
    x=alt.X("sentiment:Q", title="Sentiment score",
            scale=alt.Scale(domain=[-2.5, 2.5]),
            axis=alt.Axis(values=[-2, -1, 0, 1, 2])),
    y=alt.Y("question_short:N", title=None),
    color=alt.Color("participant_short:N", title="Participant",
                    scale=alt.Scale(scheme="tableau10")),
    tooltip=["participant_short", "question_short", "sentiment"]
)

labels = alt.Chart(strip_data).mark_text(dx=12, fontSize=10, align="left").encode(
    x=alt.X("sentiment:Q"),
    y=alt.Y("question_short:N"),
    text="participant_short:N",
    color=alt.Color("participant_short:N", legend=None,
                    scale=alt.Scale(scheme="tableau10")),
)

rule = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(
    color="gray", strokeDash=[4, 4], opacity=0.5
).encode(x="x:Q")

viz3 = (rule + points + labels).properties(
    width=500,
    height=150,
    title="Sentiment spread on most divergent questions"
)

strip_data.to_csv("viz_exports/viz3_sentiment_strip.csv", index=False)
viz3

alt.LayerChart(...)

---
## Viz 4: Theme co-occurrence heatmap

Shows which barrier themes tend to appear together in the same answers.

### Tableau instructions
1. Open `viz4_cooccurrence.csv` (already in long format)
2. Drag `theme_1` to **Rows**
3. Drag `theme_2` to **Columns**
4. Drag `count` to **Color**
5. Change mark type to **Square**
6. Color -> Edit Colors -> sequential Blue palette
7. Drag `count` to **Label**
8. Filter out rows where count = 0 (drag `count` to Filters, select range, set min to 1)
9. Sort both axes the same way (alphabetical or by total count)

In [7]:
# Deduplicate: keep only one direction (theme_1 < theme_2 alphabetically)
co_dedup = co[co["theme_1"] < co["theme_2"]].copy()
co_dedup = co_dedup[co_dedup["count"] > 0]

# Full matrix for heatmap (keep both directions)
co_full = co[co["count"] > 0].copy()

# Get theme order by total co-occurrences
theme_totals = co_full.groupby("theme_1")["count"].sum().sort_values(ascending=False)
theme_order = theme_totals.index.tolist()

chart4 = alt.Chart(co_full).mark_rect(stroke="white", strokeWidth=1).encode(
    x=alt.X("theme_2:N", sort=theme_order, title=None,
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("theme_1:N", sort=theme_order, title=None),
    color=alt.Color("count:Q",
        scale=alt.Scale(scheme="blues"),
        title="Co-occurrences"
    ),
    tooltip=["theme_1", "theme_2", "count"]
).properties(
    width=450,
    height=450,
    title="Theme co-occurrence (how often themes appear in the same answer)"
)

text4 = alt.Chart(co_full).mark_text(fontSize=10).encode(
    x=alt.X("theme_2:N", sort=theme_order),
    y=alt.Y("theme_1:N", sort=theme_order),
    text=alt.Text("count:Q"),
    color=alt.condition(
        alt.datum.count > 6,
        alt.value("white"),
        alt.value("#042C53")
    )
)

viz4 = (chart4 + text4)

co_full.to_csv("viz_exports/viz4_cooccurrence.csv", index=False)
viz4

alt.LayerChart(...)

---
## Viz 5: Participant typology scatter plot

Plots each participant by average sentiment (x) and gym confidence (y), labeled with their primary barrier. Reveals the two clusters: confident/logistical vs. struggling/psychological.

### Tableau instructions
1. Open `viz5_participant_scatter.csv`
2. Drag `avg_sentiment` to **Columns** (make sure it's a measure, continuous)
3. Drag `gym_confidence` to **Rows** (continuous)
4. Drag `participant_short` to **Label**
5. Drag `primary_barrier` to **Color**
6. Change mark type to **Circle**, increase size to ~200
7. Edit x-axis: title = 'Average sentiment', range 0 to 1
8. Edit y-axis: title = 'Gym confidence', range -1.5 to 2.5
9. Add reference lines at the median values for both axes (Analytics pane -> drag Reference Line)
10. The two clusters should be visible: top-right = confident, bottom-left = less confident
11. Add annotations to label the clusters: 'Confident, logistically blocked' and 'Less confident, seeking access'

In [8]:
pp["participant_short"] = pp["participant"].apply(short_name)

# Add jitter to gym_confidence to avoid overlap (since many are 2)
np.random.seed(42)
pp["gym_confidence_jitter"] = pp["gym_confidence"] + np.random.uniform(-0.15, 0.15, len(pp))

points5 = alt.Chart(pp).mark_circle(size=200, opacity=0.85).encode(
    x=alt.X("avg_sentiment:Q", title="Average sentiment (across all questions)",
            scale=alt.Scale(domain=[-0.1, 1.1])),
    y=alt.Y("gym_confidence_jitter:Q", title="Gym confidence (Q12 sentiment)",
            scale=alt.Scale(domain=[-1.8, 2.8])),
    color=alt.Color("primary_barrier:N", title="Primary barrier",
                    scale=alt.Scale(scheme="tableau10")),
    tooltip=["participant_short", "avg_sentiment", "gym_confidence", "primary_barrier", "top_themes"]
)

labels5 = alt.Chart(pp).mark_text(dx=12, dy=-8, fontSize=11, fontWeight=500).encode(
    x="avg_sentiment:Q",
    y="gym_confidence_jitter:Q",
    text="participant_short:N",
)

# Median reference lines
med_x = pp["avg_sentiment"].median()
med_y = pp["gym_confidence"].median()

rule_x = alt.Chart(pd.DataFrame({"x": [med_x]})).mark_rule(
    color="gray", strokeDash=[4, 4], opacity=0.4
).encode(x="x:Q")

rule_y = alt.Chart(pd.DataFrame({"y": [med_y]})).mark_rule(
    color="gray", strokeDash=[4, 4], opacity=0.4
).encode(y="y:Q")

viz5 = (rule_x + rule_y + points5 + labels5).properties(
    width=500,
    height=350,
    title="Participant typologies: sentiment vs. gym confidence"
)

pp.to_csv("viz_exports/viz5_participant_scatter.csv", index=False)
viz5

alt.LayerChart(...)

---
## Bonus: Participant theme profile grid

Small multiples showing which themes each participant exhibited.

### Tableau instructions
1. Open `viz6_theme_profiles.csv`
2. Drag `theme` to **Rows**
3. Drag `participant_short` to **Columns**
4. Drag `present` to **Color**
5. Change mark type to **Square**
6. Under Color, use a two-step palette: light gray for 0, dark teal for 1
7. This gives you a presence/absence grid for each participant

In [9]:
ptm["participant_short"] = ptm["participant"].apply(short_name)

# Order themes by total prevalence
theme_totals = ptm.groupby("theme")["present"].sum().sort_values(ascending=False)
theme_order = theme_totals.index.tolist()

viz6 = alt.Chart(ptm).mark_rect(stroke="white", strokeWidth=1).encode(
    x=alt.X("participant_short:N", title=None),
    y=alt.Y("theme:N", sort=theme_order, title=None),
    color=alt.Color("present:Q",
        scale=alt.Scale(domain=[0, 1], range=["#F1EFE8", "#1D9E75"]),
        title="Present",
        legend=None
    ),
    tooltip=["participant_short", "theme", "present"]
).properties(
    width=400,
    height=420,
    title="Theme presence by participant"
)

ptm.to_csv("viz_exports/viz6_theme_profiles.csv", index=False)
viz6

alt.Chart(...)

---
## Save all charts as PNG/SVG

In [10]:
charts = {
    "viz1_paired_barriers": chart1,
    "viz2_sentiment_heatmap": viz2,
    "viz3_sentiment_strip": viz3,
    "viz4_cooccurrence": viz4,
    "viz5_participant_scatter": viz5,
    "viz6_theme_profiles": viz6,
}

for name, chart in charts.items():
    try:
        chart.save(f"viz_exports/{name}.png", scale_factor=2)
        print(f"Saved: viz_exports/{name}.png")
    except Exception as e:
        print(f"Could not save {name}.png: {e}")
    try:
        chart.save(f"viz_exports/{name}.html")
        print(f"Saved: viz_exports/{name}.html")
    except Exception as e:
        print(f"Could not save {name}.html: {e}")

print("\nAll exports in viz_exports/")
print("CSVs are ready for Tableau, PNGs for presentations, HTMLs for interactive viewing.")

Saved: viz_exports/viz1_paired_barriers.png
Saved: viz_exports/viz1_paired_barriers.html
Saved: viz_exports/viz2_sentiment_heatmap.png
Saved: viz_exports/viz2_sentiment_heatmap.html
Saved: viz_exports/viz3_sentiment_strip.png
Saved: viz_exports/viz3_sentiment_strip.html
Saved: viz_exports/viz4_cooccurrence.png
Saved: viz_exports/viz4_cooccurrence.html
Saved: viz_exports/viz5_participant_scatter.png
Saved: viz_exports/viz5_participant_scatter.html
Saved: viz_exports/viz6_theme_profiles.png
Saved: viz_exports/viz6_theme_profiles.html

All exports in viz_exports/
CSVs are ready for Tableau, PNGs for presentations, HTMLs for interactive viewing.


In [11]:
# Generate one row per icon for Tableau pictogram
rows = []
for _, row in tp.iterrows():
    n = int(row["participants_mentioning"])
    for i in range(8):
        rows.append({
            "theme": row["theme"],
            "icon_position": i + 1,
            "filled": 1 if i < n else 0,
            "percentage": round(n / 8 * 100),
        })

pictogram_df = pd.DataFrame(rows)
pictogram_df.to_excel("viz_exports/viz_pictogram_tableau.xlsx", index=False)
print(f"Saved: {len(pictogram_df)} rows (15 themes x 8 icons each)")

Saved: 120 rows (15 themes x 8 icons each)
